# Installing the required packages

In [37]:
!pip install pandas scikit-learn joblib openpyxl

# Import the packages

In [38]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Load the uploaded Excel files

In [39]:
print("Loading datasets...")
df_clim_2024 = pd.read_excel('sri_lanka_weekly_climate_2024_plain.xlsx')
df_clim_2025 = pd.read_excel('sri_lanka_weekly_climate_2025_plain.xlsx')
df_price_2024 = pd.read_excel('sri_lanka_weekly_vegetable_prices_2024_plain.xlsx')
df_price_2025 = pd.read_excel('sri_lanka_weekly_vegetable_prices_2025_plain.xlsx')

Loading datasets...


# Concatenate years

In [40]:
df_clim = pd.concat([df_clim_2024, df_clim_2025], ignore_index=True)
df_price = pd.concat([df_price_2024, df_price_2025], ignore_index=True)

# Clean summary rows (Row 53 Annual Summaries with NaNs)

In [41]:
df_clim = df_clim.dropna(subset=['Week Start']).copy()
df_price = df_price.dropna(subset=['Week Start']).copy()

# Merge climate and price data on weekly dates

In [42]:
merged = pd.merge(df_clim, df_price, on=['Week Start', 'Week End'], suffixes=('_clim', '_price'))
merged['Week Start'] = pd.to_datetime(merged['Week Start'])

# Feature Engineering: Calendar & Cyclical Seasonality

In [43]:
merged['week_num'] = merged['Week Start'].dt.isocalendar().week.astype(int)
merged['month'] = merged['Week Start'].dt.month

**Sine and Cosine encodings model cyclic agricultural seasonality (Week 52 connects back to Week 1)**

In [44]:
merged['sin_week'] = np.sin(2 * np.pi * merged['week_num'] / 52)
merged['cos_week'] = np.cos(2 * np.pi * merged['week_num'] / 52)
print(f"Total merged weekly observations: {len(merged)}")

Total merged weekly observations: 106


# MODEL 1: Climate & Rainfall Predictor

In [45]:
print("\n--- Training Climate Model ---")
X_clim = merged[['week_num', 'month', 'sin_week', 'cos_week']]
y_clim = merged[['Total Rainfall (mm)', 'Avg Temp (°C)', 'Avg Min Temp (°C)', 'Avg Max Temp (°C)']]

climate_model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
climate_model.fit(X_clim, y_clim)


--- Training Climate Model ---


MultiOutputRegressor(estimator=RandomForestRegressor(random_state=42))

**Evaluate Climate Model**

In [46]:
clim_preds = climate_model.predict(X_clim)
print(f"Rainfall MAE: {mean_absolute_error(y_clim['Total Rainfall (mm)'], clim_preds[:, 0]):.2f} mm")
print(f"Rainfall R2:  {r2_score(y_clim['Total Rainfall (mm)'], clim_preds[:, 0]):.2f}")

joblib.dump(climate_model, 'climate_model.joblib')
print("Saved: climate_model.joblib")

Rainfall MAE: 3.73 mm
Rainfall R2:  0.95
Saved: climate_model.joblib


# MODEL 2: Crop Price Predictor (Unified across all 5 crops)

In [47]:
print("\n--- Training Price Model ---")
crop_columns = {
    'Tomato': 'Tomato (Rs./kg)',
    'Green Chilli': 'Green Chilli (Rs./kg)',
    'Cucumber': 'Cucumber (Rs./kg)',
    'Brinjal': 'Brinjal (Rs./kg)',
    'Capsicum': 'Capsicum (Rs./kg)'
}


--- Training Price Model ---


**Reshape data into long format (Crop, Week, Climate -> Price)**

In [48]:
records = []
for _, row in merged.iterrows():
    for crop_name, col_name in crop_columns.items():
        records.append({
            'crop': crop_name,
            'week_num': row['week_num'],
            'month': row['month'],
            'sin_week': row['sin_week'],
            'cos_week': row['cos_week'],
            'rainfall': row['Total Rainfall (mm)'],
            'avg_temp': row['Avg Temp (°C)'],
            'min_temp': row['Avg Min Temp (°C)'],
            'max_temp': row['Avg Max Temp (°C)'],
            'price': row[col_name]
        })

df_long = pd.DataFrame(records)

**Pipeline: One-Hot encode the crop category, pass numbers through to Random Forest**

In [49]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['crop']),
        ('num', 'passthrough', ['week_num', 'month', 'sin_week', 'cos_week', 'rainfall', 'avg_temp', 'min_temp', 'max_temp'])
    ]
)

price_pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_price = df_long[['crop', 'week_num', 'month', 'sin_week', 'cos_week', 'rainfall', 'avg_temp', 'min_temp', 'max_temp']]
y_price = df_long['price']

**Train-test split (80/20) to evaluate accuracy**

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X_price, y_price, test_size=0.2, random_state=42)
price_pipeline.fit(X_train, y_train)

test_preds = price_pipeline.predict(X_test)
print(f"Price Model Test MAE: Rs. {mean_absolute_error(y_test, test_preds):.2f} / kg")
print(f"Price Model Test R2:  {r2_score(y_test, test_preds):.2f}")

Price Model Test MAE: Rs. 82.60 / kg
Price Model Test R2:  0.74


**Re-fit pipeline on 100% of data for production**

In [51]:
price_pipeline.fit(X_price, y_price)
joblib.dump(price_pipeline, 'crop_price_model.joblib')
print("Saved: crop_price_model.joblib")

Saved: crop_price_model.joblib


**Download the files**

In [52]:
from google.colab import files
files.download('climate_model.joblib')
files.download('crop_price_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>